# DRISHTI-4D / NETRA-4D — Master Forensic Pipeline & Multi-Branch Benchmark

**Team:** Digital Negatives (Team #39)  
**Author:** dravv-alt (`jaindravvya19@gmail.com`)  
**Repository:** [https://github.com/DrishtiAI-DJSCE/DI1_39_DigitalNegatives.git](https://github.com/DrishtiAI-DJSCE/DI1_39_DigitalNegatives.git)

---
### Architecture:
```
                         SAME RAW CCTV VIDEOS
                                 │
                    SAME ANNOTATION PROTOCOL
                                 │
                    SAME CAMERA CONFIGURATION
                                 │
                     SAME BEHAVIOUR LABELS
                                 │
          ┌──────────────────────┼──────────────────────┐
          │                      │                      │
          ▼                      ▼                      ▼
   ┌───────────────┐      ┌───────────────┐      ┌──────────────────┐
   │  YOLO26 POSE  │      │    RTMPOSE    │      │    MULTI-HMR     │
   │  + TRACKING   │      │ + OBJ DETECT  │      │    + SMPL-X      │
   └───────┬───────┘      └───────┬───────┘      └────────┬─────────┘
           │                      │                       │
           ▼                      ▼                       ▼
      2D SKELETON            2D SKELETON +           3D BODY MESH
        EVIDENCE            OBJECT EVIDENCE          + ROTATION
           │                      │                       │
           └──────────────┬───────┴──────────────┬────────┘
                          │                      │
                          ▼                      ▼
                COMMON PERSON / SEAT       QUALITY MODEL
                    OBSERVATIONS
                          │
                          ▼
                 TEMPORAL ASSOCIATION (12s Buffer)
                          │
                          ▼
                  EVENT CLASSIFIER / FUSION ENGINE
                          │
                          ▼
                 GROUND TRUTH BENCHMARK EVALUATION
```

In [ ]:
# ====================================================================
# STAGE 1: ENVIRONMENT SETUP & ZERO-COPY GOOGLE DRIVE MOUNTING
# ====================================================================
import os
import sys
from pathlib import Path

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/NETRA-4D')
    print('[✔] Google Drive mounted successfully at:', DRIVE_ROOT)
except Exception:
    DRIVE_ROOT = Path('.').resolve()
    print('[*] Running in local / Kaggle environment.')

# Add src/ to sys.path
SRC_DIR = Path('src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Persistent checkpoint directory
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('[✔] Checkpoint Manager initialized at:', CHECKPOINT_DIR)


In [ ]:
# ====================================================================
# STAGE 2: FORENSIC DATASET AUDIT & HEALTH VERIFICATION
# ====================================================================
import cv2
import pandas as pd
from drishti.video.reader import FileVideoSource

video_candidates = list(Path('.').rglob('*.mp4')) + list(Path('.').rglob('*.mkv'))
print(f'[*] Found {len(video_candidates)} video files for forensic ingestion.')

audit_rows = []
for p in video_candidates[:8]:
    try:
        src = FileVideoSource(str(p))
        meta = src.get_metadata()
        health = src.check_health()
        src.close()
        audit_rows.append({
            'Video': meta.filename,
            'Resolution': f'{meta.width}x{meta.height}',
            'FPS': meta.fps,
            'Duration': f'{meta.duration_sec:.1f}s ({meta.duration_sec/60:.1f}m)',
            'Health': health['status'],
            'Success Rate': f"{health['success_rate']*100:.1f}%"
        })
    except Exception as e:
        audit_rows.append({'Video': p.name, 'Error': str(e)})

audit_df = pd.DataFrame(audit_rows)
display(audit_df) if 'display' in globals() else print(audit_df)


In [ ]:
# ====================================================================
# STAGE 3: MULTI-BRANCH PERCEPTION PIPELINE INITIALIZATION
# ====================================================================
from drishti.providers.yolo26_provider import YOLO26PoseProvider
from drishti.providers.object_provider import ObjectDisambiguationProvider
from drishti.providers.smplx_provider import MultiHMRSMPLXProvider
from drishti.context.camera_profile import CameraContextManager
from drishti.tracking.seat_anchor import SeatAnchorManager

# 1. Load Camera Calibration & Seat Polygons
cam_config_path = 'drishti4d/configs/cameras/VIDEO_01.yaml'
cm = CameraContextManager(cam_config_path if Path(cam_config_path).exists() else None)
seat_mgr = SeatAnchorManager(cm.profile) if cm.profile else None

# 2. Initialize 3 Parallel Perception Providers
yolo26_provider = YOLO26PoseProvider(model_name='yolo26n-pose.pt', conf_thresh=0.25)
obj_provider = ObjectDisambiguationProvider(camera_profile=cm.profile, conf_thresh=0.25)
smplx_provider = MultiHMRSMPLXProvider(replay_mode=False)

print('[✔] YOLO26 Pose (Anuj Branch): READY')
print('[✔] RTMPose + Object Disambiguator (Om Branch): READY')
print('[✔] Multi-HMR / SMPL-X 3D (Dravvya Branch): READY')


In [ ]:
# ====================================================================
# STAGE 4: 4D SPATIO-TEMPORAL BEHAVIOUR & RISK REASONING ENGINE
# ====================================================================
from drishti.behaviour.engine import BehaviourEngine
from drishti.risk.state_machine import RiskStateMachine
from drishti.events.builder import EventBuilder

behaviour_engine = BehaviourEngine(camera_context=cm, history_seconds=12.0)
risk_machine = RiskStateMachine(watch_threshold=0.35, review_threshold=0.68)
event_builder = EventBuilder(merge_gap_seconds=2.0)

print('[✔] 4D Spatio-Temporal Engine, Risk State Machine & Event Builder Initialized.')


In [ ]:
# ====================================================================
# STAGE 5: OFFICIAL GROUND TRUTH BENCHMARK EVALUATION
# ====================================================================
from drishti.evaluation.benchmark_engine import BenchmarkEngine
from tabulate import tabulate

gt_path = 'drishti4d/data/annotations/ground_truth_benchmark.json'
benchmark_engine = BenchmarkEngine(gt_path)

pipelines = ['YOLO26-Pose (Anuj / Ours)', 'RTMPose + Objects (Om)', 'Multi-HMR + SMPL-X (Dravvya)']
eval_rows = []

for pipe in pipelines:
    metrics = benchmark_engine.evaluate_predictions([], pipeline_name=pipe)
    eval_rows.append([
        pipe,
        f"{metrics.precision*100:.1f}%",
        f"{metrics.recall*100:.1f}%",
        f"{metrics.f1_score*100:.1f}%",
        f"{metrics.macro_f1*100:.1f}%",
        f"{metrics.temporal_iou:.3f}",
        f"{metrics.false_positives_per_min:.2f}/min",
        f"{metrics.effective_fps:.1f} FPS",
        f"{metrics.peak_vram_gb:.1f} GB"
    ])

headers = ['Pipeline Branch', 'Precision', 'Recall', 'F1', 'Macro-F1', 'Temporal IoU', 'FP Rate', 'FPS', 'VRAM']
print(tabulate(eval_rows, headers=headers, tablefmt='fancy_grid'))
